# Centralized Readmission Baseline

This notebook trains centralized models on the UCI Diabetes 130-US Hospitals dataset. It is meant to compare against the federated learning results from the Hyperledger Fabric/IPFS workflow.

Target label:

- `0 = NoEarlyReadmission` (`readmitted` is `NO` or `>30`)
- `1 = EarlyReadmission` (`readmitted` is `<30`)

Models included:

- Centralized MLP
- Centralized XGBoost

Metrics included:

- Accuracy
- Precision
- Recall
- F1 score
- AUC-ROC
- Confusion matrix

In [1]:
!pip -q install xgboost

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

## Upload Dataset

Upload `diabetic_data.csv` when prompted. The `IDS_mapping.csv` file is not required for this notebook.

In [3]:
from google.colab import files

uploaded = files.upload()
csv_name = next((name for name in uploaded.keys() if name.endswith('.csv')), None)
if csv_name is None:
    raise ValueError('Please upload diabetic_data.csv')

df_raw = pd.read_csv(csv_name)
print(df_raw.shape)
df_raw.head()

Saving diabetic_data.csv to diabetic_data.csv
(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Preprocessing

This preprocessing mirrors the project setup:

- Remove invalid gender rows
- Remove expired/hospice discharge rows
- Predict early readmission (`<30`) as the positive class
- One-hot encode categorical features
- Normalize numeric features
- Use an 80/20 train/test split

In [4]:
NUMERIC_COLUMNS = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
]

CATEGORICAL_COLUMNS = [
    'race',
    'gender',
    'age',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'max_glu_serum',
    'A1Cresult',
    'change',
    'diabetesMed',
]

MEDICATION_COLUMNS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

EXPIRED_OR_HOSPICE_DISCHARGE_IDS = {11, 13, 14, 19, 20, 21}

def diagnosis_group(value):
    if pd.isna(value) or value == '?':
        return 'missing'
    value = str(value)
    if value.startswith('V'):
        return 'supplemental'
    if value.startswith('E'):
        return 'injury'
    try:
        code = float(value)
    except ValueError:
        return 'other'
    if 390 <= code <= 459 or code == 785:
        return 'circulatory'
    if 460 <= code <= 519 or code == 786:
        return 'respiratory'
    if 520 <= code <= 579 or code == 787:
        return 'digestive'
    if 250 <= code < 251:
        return 'diabetes'
    if 800 <= code <= 999:
        return 'injury'
    if 710 <= code <= 739:
        return 'musculoskeletal'
    if 580 <= code <= 629 or code == 788:
        return 'genitourinary'
    if 140 <= code <= 239:
        return 'neoplasms'
    return 'other'

def preprocess(df):
    df = df.copy()
    df = df[df['gender'] != 'Unknown/Invalid']
    df = df[~df['discharge_disposition_id'].isin(EXPIRED_OR_HOSPICE_DISCHARGE_IDS)]
    df = df[df['readmitted'].isin(['NO', '>30', '<30'])]

    y = (df['readmitted'] == '<30').astype(np.int64).to_numpy()

    for col in ['diag_1', 'diag_2', 'diag_3']:
        df[f'{col}_group'] = df[col].map(diagnosis_group)

    feature_parts = []

    numeric = df[NUMERIC_COLUMNS].replace('?', np.nan).astype(float)
    numeric = numeric.fillna(numeric.median())
    numeric = (numeric - numeric.mean()) / numeric.std(ddof=0).replace(0, 1)
    feature_parts.append(numeric.reset_index(drop=True))

    categorical = CATEGORICAL_COLUMNS + ['diag_1_group', 'diag_2_group', 'diag_3_group'] + MEDICATION_COLUMNS
    cat_df = df[categorical].copy().replace('?', 'missing').fillna('missing').astype(str)
    cat_encoded = pd.get_dummies(cat_df, columns=categorical, dtype=np.float32)
    feature_parts.append(cat_encoded.reset_index(drop=True))

    X = pd.concat(feature_parts, axis=1).astype(np.float32)
    return X, y

X, y = preprocess(df_raw)
print('X shape:', X.shape)
print('Class counts [NoEarlyReadmission, EarlyReadmission]:', np.bincount(y))

X shape: (99340, 188)
Class counts [NoEarlyReadmission, EarlyReadmission]: [88026 11314]


In [5]:
indices = np.random.default_rng(SEED).permutation(len(y))
test_size = int(0.20 * len(y))
test_idx = indices[:test_size]
train_idx = indices[test_size:]

X_train = X.iloc[train_idx].to_numpy(dtype=np.float32)
y_train = y[train_idx]
X_test = X.iloc[test_idx].to_numpy(dtype=np.float32)
y_test = y[test_idx]

print('Train:', X_train.shape, np.bincount(y_train))
print('Test:', X_test.shape, np.bincount(y_test))

Train: (79472, 188) [70465  9007]
Test: (19868, 188) [17561  2307]


## Metric Helper

In [6]:
def compute_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc_roc': roc_auc_score(y_true, y_prob),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def print_metrics(name, metrics):
    print(name)
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1 Score : {metrics['f1']:.4f}")
    print(f"AUC-ROC  : {metrics['auc_roc']:.4f}")
    print(f"CM       : [TN:{metrics['tn']} FP:{metrics['fp']} FN:{metrics['fn']} TP:{metrics['tp']}]")

## Centralized MLP

In [7]:
class ReadmissionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.network(x)

batch_size = 256
epochs = 10
lr = 0.001

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

class_counts = np.bincount(y_train)
class_weights = len(y_train) / (2 * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

mlp = ReadmissionMLP(X_train.shape[1]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(mlp.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    mlp.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = mlp(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    print(f'Epoch {epoch:02d}/{epochs} - loss={np.mean(losses):.4f}')

Epoch 01/10 - loss=0.6605
Epoch 02/10 - loss=0.6487
Epoch 03/10 - loss=0.6455
Epoch 04/10 - loss=0.6429
Epoch 05/10 - loss=0.6398
Epoch 06/10 - loss=0.6363
Epoch 07/10 - loss=0.6325
Epoch 08/10 - loss=0.6283
Epoch 09/10 - loss=0.6245
Epoch 10/10 - loss=0.6199


In [ ]:
mlp.eval()
mlp_probs = []
mlp_preds = []
mlp_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = mlp(xb)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        mlp_probs.extend(probs)
        mlp_preds.extend(preds)
        mlp_true.extend(yb.numpy())

mlp_probs = np.array(mlp_probs)
mlp_preds = np.array(mlp_preds)
mlp_true = np.array(mlp_true)

mlp_metrics = compute_metrics(mlp_true, mlp_preds, mlp_probs)
print_metrics('Centralized MLP', mlp_metrics)
print('\nClassification report:')
print(classification_report(mlp_true, mlp_preds, target_names=['NoEarlyReadmission', 'EarlyReadmission'], zero_division=0))

## Centralized XGBoost

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
)

xgb.fit(X_train, y_train)

xgb_probs = xgb.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)

xgb_metrics = compute_metrics(y_test, xgb_preds, xgb_probs)
print_metrics('Centralized XGBoost', xgb_metrics)
print('\nClassification report:')
print(classification_report(y_test, xgb_preds, target_names=['NoEarlyReadmission', 'EarlyReadmission'], zero_division=0))

## Summary Table

In [10]:
summary = pd.DataFrame([
    {'Model': 'Centralized MLP', **mlp_metrics},
    {'Model': 'Centralized XGBoost', **xgb_metrics},
])

display_cols = ['Model', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc', 'tn', 'fp', 'fn', 'tp']
summary[display_cols].style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1': '{:.4f}',
    'auc_roc': '{:.4f}',
})

,Model,accuracy,precision,recall,f1,auc_roc,tn,fp,fn,tp
0,Centralized MLP,0.6251,0.1742,0.5960,0.2697,0.6624,11045,6516,932,1375
1,Centralized XGBoost,0.6632,0.1874,0.5696,0.2820,0.6775,11863,5698,993,1314
